In [115]:
import cv2
import numpy as np
from collections import deque
from datetime import datetime
import time

In [116]:
## os.startfile(r"C:\Users\mpoko\Workspaces\Motion-Detection\src\input-files\WildLifeExample.mp4")

## The captured video

vid = cv2.VideoCapture(r"C:\Users\mpoko\Workspaces\Motion-Detection\src\input-files\WildLifeExample.mp4")

# Check if the video was opened successfully
if not vid.isOpened():
    print("Error: Could not open video file.")
else:
    print("Video file opened successfully!")

# Read the first frame1 to confirm reading
ret1, frame1 = vid.read()
ret2,  frame2 = vid.read()
 

# else:
#     print("Error: Could not read the frame1.")

detection_zones = deque([])



Video file opened successfully!


In [117]:
# Loop until the end of the video
frame_num = 0
print(vid.get(cv2.CAP_PROP_FRAME_COUNT))
start = time.time()
while vid.isOpened():
    # Capture frame1-by-frame1
    ret1, frame1 = vid.read()

    # Stop if no frame1 is ret1urned
    ## ret1 tell the system that the video has been succeffuly read
    if not ret1:
        if vid.get(cv2.CAP_PROP_POS_FRAMES) >= vid.get(cv2.CAP_PROP_FRAME_COUNT):
            print("End of video reached")
        else:
            print("frame1 read failed unexpectedly")
        break

    frame1 = cv2.resize(frame1, (440, 280), fx = 0, fy = 0,
                         interpolation = cv2.INTER_CUBIC)
    frame2 = cv2.resize(frame2, (440, 280), fx = 0, fy = 0,
                         interpolation = cv2.INTER_CUBIC)

    # Convert to greyscale to see the difference between pixels easier
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Remove the noise in the image
    gray1 = cv2.GaussianBlur(gray1, (5, 5), 0)
    gray2 = cv2.GaussianBlur(gray2, (5, 5), 0)

    # The difference between both frame
    frame_difference = cv2.absdiff(gray1,gray2)

    # Threshold the difference
    _, frame_thresh = cv2.threshold(frame_difference, 20, 255, cv2.THRESH_BINARY)

    # Dilate to fill holes
    thresh = cv2.dilate(frame_thresh, None, iterations=12)

    # Find contours
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    motion_detected = False

    for contour in contours:
        if cv2.contourArea(contour) < 3000:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)
        motion_detected = True
    if motion_detected == True:
        current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        detection_zones.append(time.time())

        cv2.putText(
            frame1,
            f"Motion Detected: {current_timestamp}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2,
            cv2.LINE_AA
        )

        # print("Motion detected at:", timestamp)
        with open("motion_log.txt", "a") as f:
            f.write(current_timestamp + "\n")


    cv2.imshow("Motion Detection", frame1)
    cv2.imshow("Motion Detection 2", thresh)

    frame1 = frame2
    ret, frame2 = vid.read()
    frame_num += 1

    if cv2.waitKey(40) == 27:  # ESC to exit
        break

vid.release()
cv2.destroyAllWindows()
end = time.time()
length = end - start

mins,second = divmod(length,60)

print(f"Video length: {int(mins)}:{second:.2f}")
  

179.0
End of video reached
Video length: 0:6.11


In [118]:
first_detection = detection_zones.popleft()
last_detection = detection_zones.pop()



f_mins,f_sec = divmod((first_detection - start),60)

l_mins, l_sec = divmod((last_detection - start),60)

length = last_detection - first_detection
mins,second = divmod(length,60)


print(f"Movement start: {int(f_mins)}:{f_sec:.2f}")
print(f"Movement end: {int(l_mins)}:{l_sec:.2f}")
print(f"Movement length: {int(mins)}:{second:.2f}")


Movement start: 0:1.31
Movement end: 0:5.01
Movement length: 0:3.70
